# 05 · Video pipeline design / Diseño de un pipeline de vídeo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/05-video-pipeline-design.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#d97706,rgba(217,119,6,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706">PART III · GROUP · 15 MIN</span>

## Practise today / Practica hoy

Trace sampled frames to source indices and defend a sampling strategy for a brief event.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Rastrear fotogramas muestreados hasta sus índices originales y defender un muestreo para un evento breve.</div></div>

## Explore later / Explora después

Compare complete pipelines, padding costs, and synchronized-camera layouts.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Comparar pipelines completos, costes de relleno y organización de cámaras sincronizadas.</div></div>

Follow the core block immediately below. / Sigue el bloque esencial de abajo.

<!-- CORE-PATH -->
## Core path / Ruta esencial

Read and run this block from top to bottom: **recall → example → attempt → feedback → checkpoint**. Preparation and feedback definitions appear where needed. Try before opening a folded solution. Stop at **Core complete**; everything after it is **Explore later**.

🇪🇸 Lee y ejecuta este bloque de arriba abajo: **recuerda → ejemplo → intento → retroalimentación → comprobación**. La preparación y las funciones de comprobación aparecen donde se necesitan. Inténtalo antes de abrir una solución plegada. Detente en **Fin de la ruta esencial**; después empieza **Explora después**.

### Recall / Recuerda

Recall notebook 04: how would you move the colour axis of HWC to the front while keeping every pixel intact?

🇪🇸 Recuerda el cuaderno 04: ¿cómo moverías el eje de color de HWC al principio conservando cada píxel?

## Setup / Preparación

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#d97706,rgba(217,119,6,0))"></div>

This cell downloads one **real CC0 WebM video** from Wikimedia Commons, checks
its SHA-256, decodes the whole stream to count the recorded frames, and keeps
only a sparse set of real frames in memory.

The full 720-frame video is deliberately **not** materialized as one big
`(T, H, W, C)` tensor. Refusing that allocation is already a pipeline-design
decision.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Descarga un <b>video WebM real CC0</b> de Wikimedia Commons, verifica su SHA-256, decodifica el flujo completo para contar los fotogramas y conserva en memoria solo una muestra dispersa.<br><br>El video completo de 720 fotogramas no se materializa como un gran tensor <code>(T, H, W, C)</code>: evitar esa asignación ya es una decisión de diseño.</div>

### Core prep 1/1 · Preparación esencial

Run the next cell. / Ejecuta la siguiente celda.

In [ ]:
%pip install -q "imageio[ffmpeg]"

import hashlib
import io
import urllib.request

import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets

from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# Real clip: "Tormenta en l'Almadrava" by Nicolas Vigier, CC0.
# https://commons.wikimedia.org/wiki/File:Tormenta_en_l%27Almadrava.webm
VIDEO_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/1/1e/"
    "Tormenta_en_l%27Almadrava.webm"
)

VIDEO_SHA256 = (
    "e377fcdd2c79b55bce13c2c24b5dd7e412af39cd400eec548a79d0e59d79dc1b"
)

UA = "tensors-workshop/1.0 (https://github.com/project-delphi/tensors-workshop)"

def fetch_verified_video(url, expected_sha256, n_frames=16, stride=45):
    # Verify the real file, decode the full stream, retain sparse real frames.
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    raw = urllib.request.urlopen(req, timeout=120).read()

    got = hashlib.sha256(raw).hexdigest()
    if got != expected_sha256:
        raise ValueError(
            f"checksum mismatch: expected {expected_sha256}, got {got}"
        )

    kept_frames = []
    kept_source_indices = []
    total_frames = 0

    for i, frame in enumerate(
        iio.imiter(io.BytesIO(raw), plugin="FFMPEG", extension=".webm")
    ):
        total_frames = i + 1

        if i % stride == 0 and len(kept_frames) < n_frames:
            kept_frames.append(frame)
            kept_source_indices.append(i)

    clip = np.stack(kept_frames)

    return clip, np.asarray(kept_source_indices), total_frames

clip, kept_source_indices, total_frames = fetch_verified_video(
    VIDEO_URL,
    VIDEO_SHA256,
)

assert clip.shape == (16, 540, 960, 3), clip.shape
assert total_frames == 720, total_frames

print("Retained tensor / Tensor conservado:", clip.shape, clip.dtype)
print("Recorded source frames / Fotogramas grabados:", total_frames)
print(
    "Retained source indices / Índices originales conservados:",
    kept_source_indices.tolist(),
)
print(
    "RAM retained / RAM conservada:",
    f"{clip.nbytes / 1024**2:.1f} MB",
)
print()
print("EN: Setup ready with verified real video data.")
print("ES: Preparación lista con datos reales de video verificados.")

## The video axes / Los ejes del video

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#d97706,rgba(217,119,6,0))"></div>

A decoded colour video is usually `(T, H, W, C)`.

| Axis / Eje | English | Español | The question / La pregunta |
|---|---|---|---|
| `T` | time / frames | tiempo / fotogramas | Which recorded moment? / ¿Qué momento grabado? |
| `H` | height | alto | Which pixel row? / ¿Qué fila? |
| `W` | width | ancho | Which pixel column? / ¿Qué columna? |
| `C` | colour channels | canales de color | Red, green or blue? / ¿Rojo, verde o azul? |

So <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(217,119,6,.14);border:1px solid rgba(217,119,6,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(16, 540, 960, 3)</span> reads as **16 retained moments × 540 pixel
rows × 960 pixel columns × 3 colour channels**.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>(16, 540, 960, 3)</code> se lee como <b>16 momentos conservados × 540 filas de píxeles × 960 columnas × 3 canales de color</b>.</div>

### T, H, W and C, one axis at a time / T, H, W y C, un eje a la vez

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#d97706,rgba(217,119,6,0))"></div>

The table above names the four axes; this shows them. Three frames, each a tiny colour picture, and a bright dot one column further along in each. The third frame is the one to check: every channel value is 0, 128 or 255, so you can read the red plane and confirm that 255 sits exactly where the dot is. The last frame is the point of the whole section — the motion lives in `T`, and in no other axis.

$$
\text{clip}[\,t,\; h,\; w,\; c\,]
\qquad\qquad
(T, H, W, C) \longrightarrow T \cdot H \cdot W \cdot C \ \text{ numbers}
$$

Read it as: four indices, asked in that order — which moment, which row, which
column, which colour. Multiply the four sizes together and you have the count,
which is why dropping frames is the cheapest saving available and dropping
colour is the next.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-05-axes.gif" alt="An animation of a three-frame colour clip, two rows by three columns, with a red dot one column further along in each frame. One frame is then shown alone with arrows naming its height and width axes. The red channel of all three frames is shown as numbers, with 255 exactly where the dot is and 128 everywhere else. The last frame highlights the dot in each frame, at column t in frame t." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:11px">🇪🇸 ESPAÑOL</div>La tabla de arriba nombra los cuatro ejes; esto los muestra. Tres fotogramas, cada uno una imagen en color diminuta, y un punto brillante una columna más allá en cada uno. El tercer fotograma es el que conviene comprobar: todos los valores de canal son 0, 128 o 255, así que puedes leer el plano rojo y confirmar que el 255 está exactamente donde está el punto. El último fotograma es el objetivo de toda la sección: el movimiento vive en <code>T</code> y en ningún otro eje.</div>

## Exercise 1 — a real video becomes a tensor / Ejercicio 1 — un video real se vuelve tensor

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#d97706,rgba(217,119,6,0))"></div>

`clip` holds 16 real frames sampled from the verified 720-frame source.

$$
n_{\text{kept}} = \left\lceil \frac{n_{\text{total}}}{s} \right\rceil
\qquad
\text{frame } k \text{ of the clip was frame } s k \text{ of the video}
\qquad
\text{windows} = \left\lfloor \frac{n_{\text{kept}} - L}{S} \right\rfloor + 1
$$

Read it as: $s$ is the sampling stride, so keeping every $s$-th frame leaves
$n_{\text{kept}}$ of them and index $k$ no longer means what it did. $L$ is the
window length and $S$ the window stride — the two numbers somebody has to
choose before a model sees anything.


<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">PREDICT FIRST · PREDICE PRIMERO</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#d97706;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span>What does each axis of <code>(16, 540, 960, 3)</code> count?</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#d97706;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span>What percentage of the 720 recorded frames is in <code>clip</code>?</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#d97706;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span>Is <code>clip[1]</code> the original source frame <code>1</code>?</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>clip</code> tiene 16 fotogramas reales del video verificado de 720. <b>1 ·</b> ¿qué cuenta cada eje? <b>2 ·</b> ¿qué porcentaje de los 720 está presente? <b>3 ·</b> ¿<code>clip[1]</code> es el fotograma original <code>1</code>?</div>

### Optional hints / Pistas opcionales

Try first; open one hint at a time. / Inténtalo primero; abre una pista a la vez.

<details>
<summary>Hint 1 / Pista 1</summary>

Separate the number of stored frames from the number originally recorded. A retained position is not a source timestamp.

🇪🇸 Separa los fotogramas almacenados de los grabados originalmente. Una posición conservada no es una marca de tiempo original.

</details>

<details>
<summary>Hint 2 / Pista 2</summary>

Divide len(clip) by total_frames. To trace clip[k], look up kept_source_indices[k]; use 1 minus the retained fraction for the omitted fraction.

🇪🇸 Divide len(clip) entre total_frames. Para rastrear clip[k], consulta kept_source_indices[k]; usa 1 menos la fracción conservada para la omitida.

</details>



### Core activity · Actividad esencial

**Predict → Run → Explain → Check**

1. **Predict.** Does clip[1] represent source frame 1?
2. **Run.** Complete Exercise 1.
3. **Explain.** Name an event this sampling scheme could miss.
4. **Check.** Use kept_source_indices as evidence. Check the retained and discarded fractions sum to 1.

<details>
<summary>Español · Predice → Ejecuta → Explica → Comprueba</summary>

1. **Predice.** ¿clip[1] representa el fotograma original 1?
2. **Ejecuta.** Completa el Ejercicio 1.
3. **Explica.** Nombra un evento que este muestreo podría omitir.
4. **Comprueba.** Usa kept_source_indices como evidencia. Comprueba que las fracciones conservada y descartada suman 1.

</details>

Prediction / Predicción: ___  
Evidence / Evidencia: ___  
Revised explanation / Explicación revisada: ___

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# Print clip.shape and clip.dtype.
# Name axes 0, 1, 2, and 3.
#
# ES:
# Imprime clip.shape y clip.dtype.
# Nombra los ejes 0, 1, 2 y 3.
#
# TODO 2 / TAREA 2
#
# EN:
# Using len(clip) and total_frames, compute:
# - fraction of recorded frames retained
# - fraction of recorded frames not retained
#
# ES:
# Usando len(clip) y total_frames, calcula:
# - fracción de fotogramas grabados conservados
# - fracción de fotogramas grabados no conservados
#
# TODO 3 / TAREA 3
#
# EN:
# Inspect kept_source_indices.
# Explain why clip[1] is NOT source frame 1.
#
# ES:
# Inspecciona kept_source_indices.
# Explica por qué clip[1] NO es el fotograma original 1.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

retained_fraction = len(clip) / total_frames
not_retained_fraction = 1 - retained_fraction

print("Shape / Forma:", clip.shape)
print("dtype:", clip.dtype)
print("Axes / Ejes: (T, H, W, C)")
print("EN: retained time × height × width × colour.")
print("ES: tiempo conservado × alto × ancho × color.")
print()

print(
    "Retained recorded frames / Fotogramas grabados conservados:",
    f"{retained_fraction:.2%}",
)
print(
    "Not retained / No conservados:",
    f"{not_retained_fraction:.2%}",
)
print()

print(
    "clip[1] came from source frame / clip[1] proviene del fotograma original:",
    int(kept_source_indices[1]),
)

print()
print("EN: clip index and source-frame index are different coordinate systems.")
print("ES: el índice dentro de clip y el índice del video original son sistemas de coordenadas diferentes.")

fig, axes = plt.subplots(2, 1, figsize=(11, 6))

axes[0].scatter(
    np.arange(total_frames),
    np.zeros(total_frames),
    s=7,
    alpha=0.18,
    label="recorded / grabado",
)

axes[0].scatter(
    kept_source_indices,
    np.zeros_like(kept_source_indices),
    s=45,
    label="retained / conservado",
)

axes[0].set_yticks([])
axes[0].set_xlim(-5, total_frames + 5)
axes[0].set_xlabel("Source frame index / Índice del fotograma original")
axes[0].set_title(
    f"Sampling / Muestreo: {len(clip)} of/de {total_frames} "
    f"({retained_fraction:.1%})"
)
axes[0].legend(loc="upper right")

preview_slots = [0, 5, 10, 15]
strip = np.concatenate([clip[k] for k in preview_slots], axis=1)

axes[1].imshow(strip)
axes[1].set_title(
    "Four retained real frames / Cuatro fotogramas reales conservados — "
    + ", ".join(
        str(int(kept_source_indices[k])) for k in preview_slots
    )
)
axes[1].axis("off")

plt.tight_layout()
plt.show()

<details>
<summary><strong>What did Exercise 1 show? / ¿Qué mostró el Ejercicio 1?</strong></summary>

`clip` is not a small copy of the first 16 frames. It is a **sampled
sequence**: `clip[0]` is source frame `0`, `clip[1]` is source frame `45`,
`clip[2]` is source frame `90`, and so on.

Every image in it is real and measured. Most recorded moments are simply not
there.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>clip</code> no son los primeros 16 fotogramas: es una <b>secuencia muestreada</b> — <code>clip[0] → 0</code>, <code>clip[1] → 45</code>, <code>clip[2] → 90</code>. Las imágenes son reales; la mayoría de los instantes grabados no están.</div>

</details>

### Group discussion / Discusión en grupo

**Time:** 8 minutes.

Design a batch pipeline for clips of different lengths. Your task is to detect
a brief event. Each clip gets at most 16 stored frames.

- Would you sample, crop, pad, or combine them? Explain the trade-off.
- What happens when the event falls between selected frames?
- How would a model distinguish padding from a measured dark frame?

**Share:** An input-to-batch sketch with axis labels and a mask if needed.
Name one failure case and one way to test for it.

<details>
<summary>Español</summary>

**Tiempo:** 8 minutos.

Diseñen un proceso para agrupar clips de distintas duraciones. Buscan detectar
un evento breve. Pueden guardar como máximo 16 fotogramas por clip.

- ¿Muestrearían, recortarían, rellenarían o combinarían métodos? ¿Por qué?
- ¿Qué pasa si el evento ocurre entre los fotogramas seleccionados?
- ¿Cómo distinguiría el modelo el relleno de un fotograma oscuro medido?

**Compartan:** Un esquema desde la entrada hasta el lote, con ejes y una máscara
si hace falta. Nombren un posible fallo y cómo detectarlo.

</details>

### Checkpoint / Comprobación

Record your group’s sampling choice, one event it might miss, and a test that could reveal that failure.

🇪🇸 Anota la elección grupal de muestreo, un evento que podría perder y una prueba que detecte ese fallo.

Answer / Respuesta: ___

<details>
<summary>Check after attempting / Comprueba después de intentarlo</summary>

For example, fixed-interval sampling can miss an event between retained frames. Shift a known event through possible start times and measure how often it survives. Padding requires a mask.

🇪🇸 Por ejemplo, muestrear a intervalos fijos puede perder un evento entre fotogramas conservados. Desplaza un evento conocido entre posibles instantes de inicio y mide cuántas veces sobrevive. El relleno requiere una máscara.

</details>

## Core complete / Fin de la ruta esencial

Keep your prediction, evidence, and explanation. Follow the facilitator’s quiz and break schedule before continuing.

🇪🇸 Guarda tu predicción, evidencia y explicación. Sigue las pausas y quizzes del facilitador antes de continuar.

[Next: Notebook 06 / Siguiente: cuaderno 06](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/06-contraction-with-einsum.ipynb).

## Explore later / Explora después

Optional reference, exercises, and explorers. These are outside this section’s live core. Continue in order when studying them; some reuse earlier setup.

🇪🇸 Material de consulta, ejercicios y exploradores opcionales. Quedan fuera de la ruta esencial en vivo. Continúa en orden al estudiarlos; algunos reutilizan la preparación anterior.

## Why this matters / Por qué esto importa

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#d97706,rgba(217,119,6,0))"></div>

The camera recorded **720 frames**. This notebook holds **16** of them:
<span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(217,119,6,.14);border:1px solid rgba(217,119,6,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(16, 540, 960, 3)</span>.

Convenient, much smaller, and missing most of the recording.

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">THE CENTRAL IDEA · LA IDEA CENTRAL</div>Efficiency has a cost. Always ask what information was removed to make the tensor smaller or simpler.</div>

Every exercise here runs the same loop.

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">PREDICT → RUN → EXPLAIN · PREDICE → EJECUTA → EXPLICA</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#d97706;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span>Predict the shape.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#d97706;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span>Name every axis.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#d97706;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span>Run the code.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#d97706;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">4</span>Say what was kept.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#d97706;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">5</span>Say what was lost or transformed.</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La cámara grabó <b>720 fotogramas</b> y el cuaderno conserva <b>16</b>. El tensor es más pequeño y ya no contiene todos los instantes grabados.<br><br><b>La eficiencia tiene un costo: pregunta siempre qué información se eliminó.</b></div>

### What sampling throws away / Qué descarta el muestreo

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#d97706,rgba(217,119,6,0))"></div>

Keeping every other plane is cheap and lossy, and the animation shows the loss rather than describing it. After the drop, index 1 no longer means what it meant before — which is exactly why <code>clip[1]</code> is not the video's second frame.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-05-sampling.gif" alt="An animation of a tensor of three planes. One plane dims, then disappears, leaving a visible gap, and the remaining two close up into a tensor of shape 2 by 4 by 5. Index 1 now refers to what used to be plane 2." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Conservar un plano de cada dos es barato y con pérdida, y la animación muestra esa pérdida en lugar de describirla. Tras el descarte, el índice 1 ya no significa lo que significaba: por eso <code>clip[1]</code> no es el segundo fotograma del video.</div>

### Windows, not frames / Ventanas, no fotogramas

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#d97706,rgba(217,119,6,0))"></div>

What a model is handed is never the video. It is a batch of windows cut out of it, and somebody had to choose the window length and the stride. Six frames, length two, stride two: three clips, no frame shared between them.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-05-window.gif" alt="An animation of a pile of six frames with a two-frame window highlighted in turn at positions zero, two and four." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Lo que recibe un modelo nunca es el vídeo. Es un lote de ventanas recortadas de él, y alguien tuvo que elegir la longitud de la ventana y el paso. Seis fotogramas, longitud dos, paso dos: tres clips, sin ningún fotograma compartido.</div>

In [ ]:
#@title ⏸️ Step through the animations / Recorre las animaciones { display-mode: 'form' }

# Plumbing, not a lesson. The animations above loop forever and a GIF cannot
# be paused — so this fetches the same frames and hands them over one at a
# time, at whatever pace you read at.
# Plomería, no una lección: trae los mismos fotogramas y los entrega de uno en
# uno, al ritmo al que leas.

import io
import urllib.request

import ipywidgets as widgets
from IPython.display import display
from PIL import Image

gif_urls = [
    "https://project-delphi.github.io/tensors-workshop/images/cube-05-sampling.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-05-window.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-05-axes.gif",
]


def gif_frames(url):
    """Every frame of an animated GIF, as PNG bytes."""
    with urllib.request.urlopen(url, timeout=30) as response:
        gif = Image.open(io.BytesIO(response.read()))
    out = []
    try:
        while True:
            buffer = io.BytesIO()
            gif.convert("RGB").save(buffer, format="PNG")
            out.append(buffer.getvalue())
            gif.seek(gif.tell() + 1)
    except EOFError:
        pass
    return out


try:
    gif_cache = {url: gif_frames(url) for url in gif_urls}
except Exception as error:  # offline, or the site is down
    print("EN: could not reach the site, so there are no frames to step "
          "through.", error)
    print("ES: no se pudo acceder al sitio, así que no hay fotogramas que "
          "recorrer.", error)
else:
    gif_pick = widgets.Dropdown(
        options=[(url.rsplit("/", 1)[1], url) for url in gif_urls],
        description="Animation / Animación:",
        style={"description_width": "180px"},
    )
    gif_step = widgets.IntSlider(
        min=1, max=len(gif_cache[gif_urls[0]]), value=1,
        description="Frame / Fotograma:",
        style={"description_width": "180px"},
        continuous_update=False,
    )
    gif_prev = widgets.Button(description="◀ Prev")
    gif_next = widgets.Button(description="Next ▶")
    # An Image widget, deliberately, and never widgets.Output: a payload
    # leaving an Output widget makes nbclient wait out the whole cell timeout
    # (see scripts/test_notebooks.py). This one is a plain bytes trait.
    gif_view = widgets.Image(format="png",
                             layout=widgets.Layout(max_width="100%"))

    def gif_show(*_):
        frames = gif_cache[gif_pick.value]
        gif_step.max = len(frames)
        gif_view.value = frames[min(gif_step.value, len(frames)) - 1]

    def gif_bump(delta):
        def click(_):
            frames = gif_cache[gif_pick.value]
            gif_step.value = (gif_step.value - 1 + delta) % len(frames) + 1
        return click

    gif_prev.on_click(gif_bump(-1))
    gif_next.on_click(gif_bump(+1))
    gif_pick.observe(gif_show, names="value")
    gif_step.observe(gif_show, names="value")
    gif_show()

    display(widgets.VBox([
        gif_pick,
        widgets.HBox([gif_prev, gif_step, gif_next]),
        gif_view,
    ]))


### Retained-frame browser / Explorador de fotogramas conservados

Drag the slider, or press **Play**, through the 16 retained real frames.

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">WATCH TWO NUMBERS · MIRA DOS NÚMEROS</div><div style="margin:.55em 0"><code>clip[k]</code> — position inside the small tensor.</div><div style="margin:.55em 0"><code>source frame</code> — position in the original 720-frame video.</div></div>

`clip[1]` comes from source frame `45`. Not from source frame `1`.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Recorre los 16 fotogramas conservados y observa dos números: <code>clip[k]</code> es la posición dentro del tensor pequeño y <code>source frame</code> la posición en el video original. <code>clip[1]</code> viene del fotograma <code>45</code>.</div>

## Predict first / Predice primero

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#d97706,rgba(217,119,6,0))"></div>

Someone says the following. **Decide whether they are right before you
reveal anything** — commit to one answer, then open the check.

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,0.1);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">THE CLAIM · LA AFIRMACIÓN</div><div style="margin:.55em 0">&ldquo;<code>clip</code> holds 16 frames of the video, so <b><code>clip[1]</code> is the video's second frame</b>.&rdquo;</div></div>

A prediction you have committed to is worth more than one you keep
adjusting as the answer appears.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Alguien afirma que, como <code>clip</code> tiene 16 fotogramas del video, <b><code>clip[1]</code> es el segundo fotograma</b>. Decide si tiene razón <b>antes</b> de revelar la comprobación.</div>

In [ ]:
#@title 🤔 Predict: is clip[1] the video's second frame? / Predice: ¿clip[1] es el segundo fotograma del video? — run me / ejecútame { display-mode: 'form' }

# --- counterexample / contraejemplo (tested in tests/test_teaching_materials.py) ---
import numpy as np

# The retained positions: every 45th frame of the verified 720-frame source.
pred_kept = np.arange(0, 720, 45)

assert pred_kept.shape == (16,)
assert pred_kept[0] == 0
assert pred_kept[1] == 45
assert round(100 * len(pred_kept) / 720, 1) == 2.2
# --- end counterexample / fin del contraejemplo ---

import ipywidgets as widgets
from IPython.display import display

# --- how the question is laid out / cómo se presenta la pregunta ---
# Radio buttons rather than a dropdown. Four bilingual answers squeezed into
# one 640px line were hard to read, and a dropdown hides three of them until
# you open it -- the wrong shape for a question whose whole point is weighing
# the options against each other. One per line, with room around them.
# Botones de opción en vez de un desplegable: una respuesta por línea.
import contextlib
import html as pred_html
import io

PRED_ACCENT = "#d97706"
PRED_SANS = "ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"
PRED_MONO = "ui-monospace,SFMono-Regular,Menlo,Consolas,monospace"


def pred_tag(text):
    """A small EN / ES marker, in words rather than in colour alone."""
    return (f'<span style="font:700 10px/1 {PRED_MONO};letter-spacing:.16em;'
            f'color:{PRED_ACCENT};opacity:.8;margin-right:9px;'
            f'vertical-align:.12em">{text}</span>')


def pred_is_measurement(line):
    """True for a printed reading, false for a sentence.

    A reading wants monospace and tight rows so the numbers line up under one
    another; a sentence wants prose type and room. The two used to share one
    13px monospace column, which is most of why the reveal read as a wall.
    """
    if ":" not in line:
        return False
    tail = line.rsplit(":", 1)[1].strip()
    return bool(tail) and (tail[0].isdigit()
                           or tail[0] in "[(-+."
                           or tail.startswith(("True", "False", "nan", "inf")))


def pred_panel(text):
    """The reveal, laid out instead of printed.

    Exactly the same words: `check_prediction` still prints, and this catches
    what it printed and gives it typography. EN and ES stay written out as
    tags rather than becoming a colour, because a reader who cannot see the
    colour still has to be able to tell the two apart.
    """
    blocks = []
    for line in text.rstrip("\n").split("\n"):
        stripped = line.strip()
        if not stripped:
            blocks.append('<div style="height:12px"></div>')
        elif stripped.startswith(("EN:", "ES:")):
            tag, body = stripped[:2], stripped[3:].strip()
            blocks.append(
                f'<p style="margin:.55em 0;font:400 15px/1.8 {PRED_SANS}">'
                f'{pred_tag(tag)}{pred_html.escape(body)}</p>')
        elif pred_is_measurement(stripped):
            blocks.append(
                f'<div style="font:400 13.5px/2.0 {PRED_MONO};'
                f'white-space:pre-wrap">{pred_html.escape(stripped)}</div>')
        else:
            blocks.append(
                f'<p style="margin:.55em 0;font:600 15.5px/1.75 {PRED_SANS}">'
                f'{pred_html.escape(stripped)}</p>')
    return (f'<div style="border-left:4px solid {PRED_ACCENT};'
            f'background:rgba(130,130,150,.08);border-radius:0 10px 10px 0;'
            f'padding:16px 20px;margin:.4em 0 0">{"".join(blocks)}</div>')


def pred_render(choice, reveal):
    """Run the check, catch what it prints, and show it laid out."""
    caught = io.StringIO()
    with contextlib.redirect_stdout(caught):
        check_prediction(choice, reveal)
    display(widgets.HTML(pred_panel(caught.getvalue())))

pred_choice = widgets.RadioButtons(
    options=[
        ("— choose one / elige una —", None),
        ("Right — clip[1] is source frame 1 / Correcto — clip[1] es el fotograma 1", "first_sixteen"),
        ("Wrong — clip is sampled, not the first 16 / Incorrecto — clip está muestreado, no son los 16 primeros", "sampled"),
        ("Wrong — clip[1] is the last frame / Incorrecto — clip[1] es el último fotograma", "last"),
    ],
    value=None,
    description="",
    layout=widgets.Layout(width="auto", margin="0 0 6px 0"),
)

pred_reveal = widgets.Checkbox(
    value=False,
    description="Show me the answer / Muéstrame la respuesta",
    indent=False,
    layout=widgets.Layout(margin="10px 0 4px 0"),
)

def check_prediction(choice, reveal):
    if choice is None:
        print("Choose an answer first / Elige una respuesta primero.")
        return

    if not reveal:
        print("Answer saved / Respuesta guardada.")
        print("Tick the box above when you are ready / Marca la casilla de arriba\n      cuando quieras.".replace("\n      ", " "))
        return

    print("Retained positions / Posiciones conservadas:", pred_kept)
    print()
    print("clip[0] is source frame / viene del fotograma:", pred_kept[0])
    print("clip[1] is source frame / viene del fotograma:", pred_kept[1])
    print("Fraction of recorded moments / Fracción de instantes:",
          f"{100 * len(pred_kept) / 720:.1f}%")
    print()
    if choice == "sampled":
        print("You were right / Acertaste.")
    else:
        print("You were wrong — read on / Te equivocaste; sigue leyendo.")
    print()
    print("EN: clip is a sampled sequence, not a copy of the first 16 frames. clip[1] is source frame 45, and about 97.8% of the recorded moments are simply not in the tensor.")
    print("ES: clip es una secuencia muestreada, no una copia de los 16 primeros fotogramas. clip[1] viene del fotograma 45, y cerca del 97,8% de los instantes grabados no están en el tensor.")

# The one <style> block in these notebooks, and the markdown rule does not
# cover it. ipywidgets gives no way to set the space between radio options
# from Python, and this is *widget output*, not a markdown cell: Colab strips
# <style> from markdown -- which is why every box in these notebooks is
# inline-styled -- but renders it in an output, the same path pandas' own
# Styler uses. Scoped to one added class so it can reach nothing else, and if
# it is ever dropped the options still work, just closer together.
pred_choice.add_class("pred-radio")

display(widgets.HTML(
    "<style>"
    ".pred-radio .widget-radio-box label{display:flex;align-items:flex-start;"
    "margin:0 0 13px;font:400 15px/1.6 " + PRED_SANS + "}"
    ".pred-radio input[type=radio]{flex:none;margin:4px 11px 0 0;"
    "transform:scale(1.15)}"
    "</style>"
))

pred_output = widgets.interactive_output(
    pred_render,
    {"choice": pred_choice, "reveal": pred_reveal},
)

pred_heading = widgets.HTML(
    f'<div style="font:700 11px/1.6 {PRED_MONO};letter-spacing:.18em;'
    f'color:{PRED_ACCENT};margin:2px 0 12px">'
    f'YOUR PREDICTION \u00b7 TU PREDICCI\u00d3N</div>'
)

display(widgets.VBox(
    [pred_heading, pred_choice, pred_reveal, pred_output],
    layout=widgets.Layout(padding="2px 0 14px 0"),
))

In [ ]:
#@title 🎞️ Retained-frame browser / Explorador de fotogramas — run me / ejecútame { display-mode: 'form' }

frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(clip) - 1,
    step=1,
    description="clip[k]:",
    continuous_update=False,
    style={"description_width": "80px"},
)

frame_play = widgets.Play(
    value=0,
    min=0,
    max=len(clip) - 1,
    step=1,
    interval=500,
    description="Play",
)

widgets.jslink(
    (frame_play, "value"),
    (frame_slider, "value"),
)

def show_retained_frame(k):
    plt.close("all")

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.imshow(clip[k])
    ax.set_title(
        f"clip[{k}] → source frame / fotograma original "
        f"{int(kept_source_indices[k])} of/de {total_frames}"
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print("Tensor position / Posición en tensor:", k)
    print("Source position / Posición original:", int(kept_source_indices[k]))
    print("Frame shape / Forma del fotograma:", clip[k].shape)

frame_output = widgets.interactive_output(
    show_retained_frame,
    {"k": frame_slider},
)

display(
    widgets.VBox([
        widgets.HBox([frame_play, frame_slider]),
        frame_output,
    ])
)

### Sampling calculator / Calculadora de muestreo

The real tensor keeps 16 frames. This asks a design question instead:

**if I kept a different number of positions from a 720-frame video, what
fraction of recorded timesteps would that be?**

It changes only the sampling *plan*. It decodes nothing and invents nothing.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El tensor real conserva 16 fotogramas. Esto plantea otra pregunta: si conservaras otro número de posiciones de un video de 720, ¿qué fracción de los instantes sería? Solo cambia el <b>plan</b>: no decodifica ni inventa fotogramas.</div>

In [ ]:
#@title 🧮 Sampling calculator / Calculadora de muestreo — run me / ejecútame { display-mode: 'form' }

keep_slider = widgets.IntSlider(
    value=16,
    min=1,
    max=120,
    step=1,
    description="Keep / Conservar:",
    continuous_update=False,
    style={"description_width": "120px"},
)

def sampling_calculator(n_keep):
    fraction = n_keep / total_frames
    not_kept = 1 - fraction

    # Evenly spaced design positions for visualization only.
    planned = np.linspace(
        0,
        total_frames - 1,
        n_keep,
        dtype=int,
    )

    plt.close("all")
    fig, ax = plt.subplots(figsize=(10, 1.8))

    ax.scatter(
        np.arange(total_frames),
        np.zeros(total_frames),
        s=5,
        alpha=0.12,
    )

    ax.scatter(
        planned,
        np.zeros_like(planned),
        s=28,
    )

    ax.set_yticks([])
    ax.set_xlim(-5, total_frames + 5)
    ax.set_xlabel("Recorded frame index / Índice de fotograma grabado")
    ax.set_title(
        f"Design plan / Plan de diseño: {n_keep} of/de {total_frames}"
    )

    plt.tight_layout()
    plt.show()

    print(f"Retained fraction / Fracción conservada: {fraction:.2%}")
    print(f"Not retained / No conservada: {not_kept:.2%}")
    print("EN: fewer retained positions reduce memory but observe time more sparsely.")
    print("ES: menos posiciones conservadas reducen memoria, pero observan el tiempo de forma más dispersa.")

sampling_output = widgets.interactive_output(
    sampling_calculator,
    {"n_keep": keep_slider},
)

display(widgets.VBox([keep_slider, sampling_output]))

## Exercise 2 — one input, two goals / Ejercicio 2 — una entrada, dos objetivos

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#d97706,rgba(217,119,6,0))"></div>

Both systems start from `(T, H, W, C)`. They are asking different questions.

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">SYSTEM A · SISTEMA A — short-video recommender</div><b>“What is this whole video about?”</b><div style='margin-top:10px'>One embedding per video: <code>(N, embedding)</code>. Time is summarized.</div></div>

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">SYSTEM B · SISTEMA B — surgical phase labelling</div><b>“What phase is happening at each timestep?”</b><div style='margin-top:10px'>A prediction per moment: <code>(N, T, classes)</code>. Time must survive.</div></div>

These are design scenarios, not new datasets.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Ambos parten de <code>(T, H, W, C)</code> y preguntan cosas distintas. Un recomendador necesita <b>una representación de todo el video</b>; un sistema de fases quirúrgicas necesita una predicción <b>en cada instante</b>, así que el eje temporal debe sobrevivir. Son escenarios de diseño, no conjuntos de datos nuevos.</div>

In [ ]:
# TODO 4 / TAREA 4
#
# EN:
# Fill in one defensible shape at each stage for BOTH systems.
# Next to every shape, write what each axis means.
#
# --- Tech: short-video recommender -------------------------------------------
# raw file          : ...
# decoded frames    : ...
# preprocessed batch: ...
# model input       : ...
# model output      : ...
#
# --- Biotech: surgical phase labelling --------------------------------------
# raw file          : ...
# decoded frames    : ...
# preprocessed batch: ...
# model input       : ...
# model output      : ...
#
# Then answer:
# Which system intentionally removes the time axis at the output?
#
# ES:
# Completa una forma coherente en cada etapa para AMBOS sistemas.
# Junto a cada forma, escribe qué significa cada eje.
#
# Después responde:
# ¿Qué sistema elimina intencionalmente el eje temporal en su salida?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

# One defensible design. Other sizes can also be correct when the
# axis meanings and system goals remain internally consistent.

# --- Tech: short-video recommender -------------------------------------------
# raw file          : bytes on disk / bytes en disco
# decoded frames    : (T, H, W, C)
# preprocessed batch: (32, 8, 224, 224, 3)  -> N, T, H, W, C
# model input       : (32, 8, 224, 224, 3)
# model output      : (32, 512)              -> N, embedding
# Time is summarized into one representation per video.

# --- Biotech: surgical phase labelling ---------------------------------------
# raw file          : bytes on disk / bytes en disco
# decoded frames    : (T, H, W, C)
# preprocessed batch: (4, 64, 224, 224, 3)  -> N, T, H, W, C
# model input       : (4, 64, 224, 224, 3)
# model output      : (4, 64, 12)            -> N, T, classes
# Time remains because the predicted phase may change at each timestep.

print("Real anchor / Ancla real:", clip.shape, "-> (T, H, W, C)")
print()
print("Tech output / Salida tech:", (32, 512))
print("EN: one vector per video; time has been summarized.")
print("ES: un vector por video; el tiempo ha sido resumido.")
print()
print("Biotech output / Salida biotech:", (4, 64, 12))
print("EN: one class distribution per timestep; time remains.")
print("ES: una distribución de clases por instante; el tiempo permanece.")

### Does time survive? / ¿Sobrevive el tiempo?

Switch between the two systems.

The question is not which shape is better. It is: **does the task need one
answer for the whole video, or an answer at every timestep?**

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La pregunta no es qué forma es «mejor», sino: ¿la tarea necesita una respuesta para todo el video o una por cada instante?</div>

In [ ]:
#@title 🔀 Does time survive? / ¿Sobrevive el tiempo? — run me / ejecútame { display-mode: 'form' }

system_toggle = widgets.ToggleButtons(
    options=[
        ("Recommender / Recomendador", "recommender"),
        ("Surgical phases / Fases quirúrgicas", "surgical"),
    ],
    value="recommender",
    description="System / Sistema:",
    style={"description_width": "115px"},
)

def explain_system(system):
    if system == "recommender":
        stages = [
            ("Decoded / Decodificado", "(T,H,W,C)"),
            ("Batch", "(N,T,H,W,C)"),
            ("Output / Salida", "(N,embedding)"),
        ]
        en = "Time is summarized because the output describes the whole video."
        es = "El tiempo se resume porque la salida describe el video completo."
        survives = "NO"
    else:
        stages = [
            ("Decoded / Decodificado", "(T,H,W,C)"),
            ("Batch", "(N,T,H,W,C)"),
            ("Output / Salida", "(N,T,classes)"),
        ]
        en = "Time survives because the output can change at every timestep."
        es = "El tiempo permanece porque la salida puede cambiar en cada instante."
        survives = "YES / SÍ"

    print("Pipeline / Pipeline")
    for label, shape in stages:
        print(f"{label:22s} → {shape}")

    print()
    print("Time survives? / ¿Sobrevive el tiempo?:", survives)
    print("EN:", en)
    print("ES:", es)

system_output = widgets.interactive_output(
    explain_system,
    {"system": system_toggle},
)

display(widgets.VBox([system_toggle, system_output]))

<details>
<summary><strong>Why can both designs be correct? / ¿Por qué ambos diseños pueden ser correctos?</strong></summary>

A tensor shape is part of the **task definition**.

Whole-video classification or retrieval may summarize time on purpose.
Timestep labelling cannot: dropping time destroys the coordinate that attaches
a prediction to a moment.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La forma del tensor es parte de la <b>definición de la tarea</b>. Clasificar un video entero puede resumir el tiempo a propósito; etiquetar por instante no puede, porque eliminar el tiempo destruye la coordenada que ata una predicción a un momento.</div>

</details>

## Exercise 3 — variable-length clips: pad or sample? / Ejercicio 3 — clips de longitud variable: ¿padding o muestreo?

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#d97706,rgba(217,119,6,0))"></div>

Real systems get videos of different durations. To make the memory trade-off
visible **without allocating a gigantic tensor**, we use a stress-test
scenario: 30 seconds, 45 seconds, 2 minutes and 4 hours, all at 30 fps.

Those durations are chosen design inputs, not measurements of the Wikimedia
video.

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">TWO OPTIONS · DOS OPCIONES</div><div style="margin:.55em 0"><b>A · Padding</b> — stretch every sequence to the longest, and mask which positions are real.</div><div style="margin:.55em 0"><b>B · Fixed sampling</b> — keep exactly 64 positions from every clip. Predictable memory; long clips observed far more sparsely.</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Escenario de estrés: 30 s, 45 s, 2 min y 4 h a 30 fps — entradas de diseño elegidas, no mediciones. <b>Padding:</b> todas las secuencias hasta la más larga, con máscara. <b>Muestreo fijo:</b> 64 posiciones de cada clip; memoria predecible, videos largos mucho más dispersos.</div>

In [ ]:
# TODO 5 / TAREA 5
#
# EN:
# Convert [30 s, 45 s, 2 min, 4 h] at 30 fps into frame counts.
# If all four are padded to the longest sequence:
# - what is the mask shape?
# - what fraction of (N,T) positions are padding?
#
# ES:
# Convierte [30 s, 45 s, 2 min, 4 h] a 30 fps en cantidades de fotogramas.
# Si las cuatro secuencias se rellenan hasta la más larga:
# - ¿cuál es la forma de la máscara?
# - ¿qué fracción de posiciones (N,T) corresponde a padding?
#
# TODO 6 / TAREA 6
#
# EN:
# Compare padding with keeping exactly 64 positions from every clip.
# What is gained? What temporal detail may be missed?
#
# ES:
# Compara padding con conservar exactamente 64 posiciones de cada clip.
# ¿Qué se gana? ¿Qué detalle temporal puede perderse?
#
# TODO 7 / TAREA 7
#
# EN:
# A surgical system has 3 synchronized cameras.
# Write:
# - one shape with camera as its own CAM axis;
# - one shape that folds camera into the batch axis.
# When must CAM remain explicit?
#
# ES:
# Un sistema quirúrgico tiene 3 cámaras sincronizadas.
# Escribe:
# - una forma con cámara como eje CAM independiente;
# - una forma donde CAM se integra en el eje de lote.
# ¿Cuándo debe mantenerse CAM explícito?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

durations_s = np.array([
    30,
    45,
    2 * 60,
    4 * 60 * 60,
])

fps = 30
lengths = durations_s * fps
T_max = int(lengths.max())

mask = np.zeros(
    (len(lengths), T_max),
    dtype=bool,
)

for i, n_frames in enumerate(lengths):
    mask[i, :int(n_frames)] = True

valid_fraction = mask.sum() / mask.size
padding_fraction = 1 - valid_fraction

print("Frame counts / Cantidad de fotogramas:", lengths.tolist())
print("Mask shape / Forma de máscara:", mask.shape)
print(f"Measured positions / Posiciones medidas: {valid_fraction:.2%}")
print(f"Padding positions / Posiciones de padding: {padding_fraction:.2%}")
print()

fig, ax = plt.subplots(figsize=(10, 2.8))

ax.imshow(
    mask,
    aspect="auto",
    cmap="Greys",
    interpolation="nearest",
)

ax.set_yticks(range(4))
ax.set_yticklabels([
    "30 s",
    "45 s",
    "2 min",
    "4 h",
])

ax.set_xlabel("Frame position T / Posición temporal T")
ax.set_title(
    f"Valid vs padding / Válido vs padding — "
    f"{padding_fraction:.2%} padding"
)

plt.tight_layout()
plt.show()

sampled_batch_shape = (4, 64, 224, 224, 3)

print("Fixed-sampling batch / Lote con muestreo fijo:", sampled_batch_shape)
print("EN: memory becomes bounded and predictable.")
print("ES: la memoria se vuelve acotada y predecible.")
print("EN: a long clip is represented by much more widely spaced observations.")
print("ES: un clip largo queda representado por observaciones mucho más separadas.")
print()

explicit_camera = (4, 3, 64, 224, 224, 3)  # N, CAM, T, H, W, C
folded_camera = (12, 64, 224, 224, 3)      # N*CAM, T, H, W, C

print("Camera explicit / Cámara explícita:", explicit_camera)
print("Axes / Ejes: (N, CAM, T, H, W, C)")
print()
print("Camera folded / Cámara integrada:", folded_camera)
print("Axes / Ejes: (N*CAM, T, H, W, C)")
print()
print("EN: keep CAM explicit when the model must know which synchronized view a frame came from or combine views structurally.")
print("ES: conserva CAM explícito cuando el modelo debe saber de qué vista sincronizada proviene un fotograma o combinar las vistas de forma estructurada.")

In [ ]:
# Exercise 3 design scenario, recomputed here in a visible cell so the three
# explorers below run whether or not the folded solution was executed. The
# valid-vs-padding heatmap, the printed step-by-step percentages, and the
# pad-vs-sample / camera-axis reasoning stay folded in the solution above.
durations_s = np.array([30, 45, 2 * 60, 4 * 60 * 60])
fps = 30
lengths = durations_s * fps
T_max = int(lengths.max())
padding_fraction = 1 - lengths.sum() / (len(lengths) * T_max)

sampled_batch_shape = (4, 64, 224, 224, 3)   # N, T, H, W, C
explicit_camera = (4, 3, 64, 224, 224, 3)    # N, CAM, T, H, W, C
folded_camera = (12, 64, 224, 224, 3)        # N*CAM, T, H, W, C

### Duration and padding explorer / Explorador de duración y padding

Pick one of the four durations. You get its frame positions at 30 fps, the
padding it would receive in a batch padded to 4 hours, and how sparse a
64-position plan would be.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige una de las cuatro duraciones: posiciones de fotograma a 30 fps, el padding en un lote rellenado hasta 4 horas, y qué tan disperso sería un plan de 64 posiciones.</div>

In [ ]:
#@title ⏱️ Duration explorer / Explorador de duración — run me / ejecútame { display-mode: 'form' }

duration_selector = widgets.Dropdown(
    options=[
        ("30 seconds / 30 segundos", 0),
        ("45 seconds / 45 segundos", 1),
        ("2 minutes / 2 minutos", 2),
        ("4 hours / 4 horas", 3),
    ],
    value=0,
    description="Clip:",
    style={"description_width": "80px"},
)

def explore_duration(index):
    frames = int(lengths[index])
    padding = T_max - frames
    padding_for_clip = padding / T_max
    spacing = frames / 64

    print("Measured frame positions / Posiciones medidas:", frames)
    print("Padded to / Rellenado hasta:", T_max)
    print("Padding positions / Posiciones de padding:", padding)
    print(f"Padding fraction for this clip / Fracción de padding: {padding_for_clip:.2%}")
    print()

    print("If 64 positions are sampled / Si se muestrean 64 posiciones:")
    print(f"Average spacing / Separación media aproximada: {spacing:.1f} recorded frames")
    print()
    print("EN: padding preserves all measured positions but can waste representation space.")
    print("ES: el padding conserva todas las posiciones medidas, pero puede desperdiciar espacio de representación.")
    print("EN: fixed sampling bounds memory but observes long videos more sparsely.")
    print("ES: el muestreo fijo limita la memoria, pero observa los videos largos de forma más dispersa.")

duration_output = widgets.interactive_output(
    explore_duration,
    {"index": duration_selector},
)

display(widgets.VBox([duration_selector, duration_output]))

### Pad or sample? / ¿Padding o muestreo?

Switch between the strategies. There is no universally right answer — only
whether keeping every measured timestep matters more than bounding memory and
compute.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>No hay respuesta universalmente correcta: depende de si conservar todos los instantes medidos importa más que limitar memoria y cómputo.</div>

In [ ]:
#@title ⚖️ Pad or sample? / ¿Padding o muestreo? — run me / ejecútame { display-mode: 'form' }

strategy_toggle = widgets.ToggleButtons(
    options=[
        ("Padding", "padding"),
        ("Fixed 64 samples / 64 muestras fijas", "sampling"),
    ],
    value="padding",
    description="Strategy / Estrategia:",
    style={"description_width": "135px"},
)

def explain_strategy(strategy):
    if strategy == "padding":
        print("Shape concept / Concepto de forma: (N, T_max, H, W, C)")
        print("Mask / Máscara: (N, T_max)")
        print(f"Stress-test padding / Padding del escenario: {padding_fraction:.2%}")
        print("EN: measured timesteps are preserved, but padded slots are not observations.")
        print("ES: se conservan los instantes medidos, pero las posiciones de padding no son observaciones.")
    else:
        print("Example batch / Lote de ejemplo:", sampled_batch_shape)
        print("EN: every clip contributes exactly 64 sampled positions.")
        print("ES: cada clip aporta exactamente 64 posiciones muestreadas.")
        print("EN: memory is predictable, but events between sampled positions may be missed.")
        print("ES: la memoria es predecible, pero pueden perderse eventos entre posiciones muestreadas.")

strategy_output = widgets.interactive_output(
    explain_strategy,
    {"strategy": strategy_toggle},
)

display(widgets.VBox([strategy_toggle, strategy_output]))

### Camera-axis explorer / Explorador del eje de cámaras

Synchronized cameras add a design choice: <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(217,119,6,.14);border:1px solid rgba(217,119,6,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(N, CAM, T, H, W, C)</span> or
<span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(217,119,6,.14);border:1px solid rgba(217,119,6,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(N×CAM, T, H, W, C)</span>.

Pick one and read what it costs.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Varias cámaras sincronizadas añaden una decisión: <code>(N, CAM, T, H, W, C)</code> o <code>(N×CAM, T, H, W, C)</code>. Elige una y observa qué cuesta.</div>

In [ ]:
#@title 🎥 Camera-axis explorer / Explorador del eje de cámaras — run me / ejecútame { display-mode: 'form' }

camera_toggle = widgets.ToggleButtons(
    options=[
        ("Keep CAM explicit / Mantener CAM explícito", "explicit"),
        ("Fold CAM into batch / Integrar CAM al lote", "folded"),
    ],
    value="explicit",
    description="Camera / Cámara:",
    style={"description_width": "120px"},
)

def explain_camera(choice):
    if choice == "explicit":
        print("Shape / Forma:", explicit_camera)
        print("Axes / Ejes: (N, CAM, T, H, W, C)")
        print("EN: camera identity remains a separate coordinate.")
        print("ES: la identidad de la cámara permanece como una coordenada independiente.")
        print("EN: useful when synchronized views must be combined or compared.")
        print("ES: útil cuando las vistas sincronizadas deben combinarse o compararse.")
    else:
        print("Shape / Forma:", folded_camera)
        print("Axes / Ejes: (N*CAM, T, H, W, C)")
        print("EN: camera identity is no longer represented by a dedicated axis.")
        print("ES: la identidad de la cámara ya no está representada por un eje dedicado.")
        print("EN: this may be acceptable only if treating views as separate examples matches the task.")
        print("ES: puede ser adecuado solo si tratar las vistas como ejemplos separados coincide con la tarea.")

camera_output = widgets.interactive_output(
    explain_camera,
    {"choice": camera_toggle},
)

display(widgets.VBox([camera_toggle, camera_output]))

## What just happened / Qué acaba de pasar

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#d97706,rgba(217,119,6,0))"></div>

You followed one real, verified video from a file to a tensor, then used that
concrete case to reason about much larger systems.

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">FIVE IDEAS · CINCO IDEAS</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#d97706;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span><b>A video file is not yet a tensor.</b> It has to be decoded into frames and axes.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#d97706;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span><b>Sampling changes what is observed.</b> This pipeline kept 16 of 720 frames — about <b>2.2%</b> of the recorded moments.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#d97706;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span><b>The output shape depends on the question.</b> Summarize time, or preserve it.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#d97706;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">4</span><b>Padding and sampling solve different problems.</b> One keeps measured positions and wastes space; the other bounds memory and observes sparsely.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#d97706;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">5</span><b>Camera, batch and time are different axes.</b> Not interchangeable for being dimensions.</div></div>

A good video pipeline makes every axis, and every loss of information,
explicit.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>1 ·</b> un archivo de video todavía no es un tensor; <b>2 ·</b> el muestreo cambia lo observado — 16 de 720, un 2,2 %; <b>3 ·</b> la forma de salida depende de la pregunta; <b>4 ·</b> padding y muestreo resuelven problemas distintos; <b>5 ·</b> cámara, lote y tiempo son ejes diferentes.<br><br>Un buen pipeline hace explícito cada eje y cada pérdida de información.</div>

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#d97706,rgba(217,119,6,0))"></div>

## Done with this section / Fin de esta sección

Next / Siguiente: **06 · Contraction with einsum / Contracción con einsum** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/06-contraction-with-einsum.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)